# Silver → Gold

Nesta etapa, vamos transformar as tabelas tratadas da Silver em um modelo organizado para consultas de negócio. Construiremos as dimensões, os relacionamentos entre filmes e suas entidades e uma tabela fato com as métricas dos filmes lançados. Também prepararemos a tabela de contexto solicitada para o futuro assistente de IA, sem implementar o assistente. Ao final, responderemos às seis perguntas da atividade. As limitações de cobertura e qualidade identificadas na Silver serão consideradas nas junções e na interpretação dos resultados.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime
from zoneinfo import ZoneInfo

catalogo = "workspace"
schema_silver = "silver"
schema_gold = "gold"

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Uma única data para as verificações desta execução.
data_execucao = datetime.now(
    ZoneInfo("America/Recife")
).date()

nomes_fontes = [
    "tb_info_filmes",
    "tb_cotacao_dolar",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
]

# Conferimos todas as fontes antes de iniciar a preparação.
fontes_ausentes = [
    nome
    for nome in nomes_fontes
    if not spark.catalog.tableExists(
        f"{catalogo}.{schema_silver}.{nome}"
    )
]

if fontes_ausentes:
    raise ValueError(
        "Tabelas Silver não encontradas: "
        + ", ".join(fontes_ausentes)
    )

fontes_silver = {
    nome: spark.table(f"{catalogo}.{schema_silver}.{nome}")
    for nome in nomes_fontes
}

resumo_fontes = [
    (nome, df.count())
    for nome, df in fontes_silver.items()
]

display(
    spark.createDataFrame(
        resumo_fontes,
        ["tabela_silver", "total_registros"],
    )
)

if any(total == 0 for _, total in resumo_fontes):
    raise ValueError(
        "Existe uma fonte Silver vazia. "
        "Confira a execução anterior antes de continuar."
    )

print("Data de execução:", data_execucao)
print("As sete fontes Silver estão disponíveis.")

tabela_silver,total_registros
tb_info_filmes,97594
tb_cotacao_dolar,12
tb_financeiro_filmes,99006
tb_metricas_engajamento,98197
tb_avaliacoes_usuarios,32412
tb_generos,141356
tb_pessoas_empresas,891652


Data de execução: 2026-09-21
As sete fontes Silver estão disponíveis.


#### Resultado da leitura das fontes

As sete tabelas Silver estão disponíveis, e suas quantidades coincidem com a conferência realizada ao final da camada anterior. A leitura foi feita diretamente das tabelas salvas, sem executar novamente os notebooks anteriores. Os resultados numéricos documentados neste notebook correspondem à execução de 20/09/2026. Uma nova execução poderá alterar os resultados caso as fontes ou os parâmetros de referência sejam atualizados.

## Dimensão de filmes

Vamos construir dim_movies com os atributos descritivos exigidos na atividade. Manteremos id_filme para rastrear o registro até a origem e criaremos sk_movie_id como chave numérica para os relacionamentos da Gold. A dimensão conterá todos os filmes disponíveis em tb_info_filmes; o filtro de filmes lançados será aplicado na construção da tabela fato. 

Usaremos uma numeração ordenada pelo identificador original, que produz as mesmas chaves para o mesmo conjunto de filmes. Como a inclusão de filmes pode alterar essa numeração, as dimensões e suas tabelas relacionadas deverão ser reconstruídas juntas antes de consumir a Gold.

A numeração global utilizada para gerar as chaves exige uma etapa de processamento em uma única partição, conforme o aviso emitido pelo Spark. Mantive essa estratégia pela simplicidade e pela ordenação reproduzível para o mesmo conjunto de dados. Para volumes maiores, a geração e a persistência das chaves precisariam ser revistas.

In [0]:
df_info_silver = fontes_silver["tb_info_filmes"]

# A dimensão precisa ter uma identificação preenchida
# e apenas um registro por filme.
ids_invalidos = (
    df_info_silver
    .filter(
        F.col("id_filme").isNull()
        | (F.trim("id_filme") == "")
    )
    .limit(1)
    .count()
)

ids_repetidos = (
    df_info_silver
    .groupBy("id_filme")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if ids_invalidos or ids_repetidos:
    raise ValueError(
        "tb_info_filmes possui identificadores ausentes "
        "ou repetidos. Revise a Silver."
    )

# A ordenação usa a chave natural única.
janela_chave_filme = Window.orderBy("id_filme")

df_dim_movies = (
    df_info_silver
    .withColumn(
        "sk_movie_id",
        F.row_number().over(janela_chave_filme).cast("long"),
    )
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse",
    )
)

display(
    df_dim_movies.agg(
        F.count("*").alias("total_filmes"),
        F.countDistinct("id_filme").alias("ids_origem_distintos"),
        F.countDistinct("sk_movie_id").alias("chaves_gold_distintas"),
    )
)

display(
    df_dim_movies
    .groupBy("status_filme")
    .agg(F.count("*").alias("quantidade_filmes"))
    .orderBy(F.desc("quantidade_filmes"))
)

display(
    df_dim_movies
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "status_filme",
    )
    .orderBy("sk_movie_id")
    .limit(10)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


total_filmes,ids_origem_distintos,chaves_gold_distintas
97594,97594,97594


status_filme,quantidade_filmes
Lançado,96241
Pós-Produção,700
Em Produção,604
Planejado,47
Não Informado,2


sk_movie_id,id_filme,titulo,data_lancamento,status_filme
1,1000004,Purple Beatz,2022-07-07,Lançado
2,1000005,Aisha Brown: The First Black Woman Ever,2020-02-14,Lançado
3,1000007,KYLE BROWNRIGG: INTRODUCING LYLE,2022-05-27,Lançado
4,1000011,Worth Your Weight in Gold,2022-07-14,Lançado
5,1000014,On va manquer !,2018-05-15,Lançado
6,1000030,58 Hours: The Baby Jessica Story,2021-07-31,Lançado
7,1000054,One Hundred Years and Hope,2022-06-18,Lançado
8,1000058,Homecoming,2023-07-12,Lançado
9,1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,Lançado
10,1000073,A Chance To Win,2023-05-03,Lançado


#### Resultado da dimensão de filmes

A dimensão contém 97.594 filmes, com identificadores de origem e chaves Gold únicos. A distribuição apresenta 96.241 filmes com status Lançado, 700 em Pós-Produção, 604 em Produção, 47 Planejados e dois com status Não Informado. 

Todos permanecem na dimensão para preservar seus atributos descritivos; a tabela fato receberá o filtro de filmes lançados. A unicidade das chaves confirma a estrutura do relacionamento, mas não elimina as limitações de conteúdo identificadas na Silver.

## Dimensões de gêneros, pessoas e produtoras

Vamos construir os cadastros únicos de gêneros, pessoas e produtoras. Neste momento, separaremos os nomes dos vínculos com os filmes; esses vínculos serão recuperados nas tabelas de relacionamento da próxima etapa. Na dimensão de pessoas, a identificação será formada pelo nome e pelo tipo de atuação, permitindo que uma mesma pessoa apareça como Ator e Diretor. 

Como a origem não fornece identificadores individuais dessas entidades, nomes iguais dentro do mesmo tipo serão agrupados, o que pode reunir pessoas diferentes com mesmo nome. As chaves serão numéricas e geradas com ordenação explícita, seguindo a mesma estratégia adotada na dimensão de filmes.

In [0]:
df_generos_silver = fontes_silver["tb_generos"]
df_entidades_silver = fontes_silver["tb_pessoas_empresas"]

# Cada gênero terá uma única entrada no cadastro.
df_dim_genres = (
    df_generos_silver
    .select("nome_genero")
    .distinct()
    .withColumn(
        "sk_genre_id",
        F.row_number()
        .over(Window.orderBy("nome_genero"))
        .cast("long"),
    )
    .select("sk_genre_id", "nome_genero")
)

# O cadastro de pessoas distingue os tipos de atuação.
df_dim_people = (
    df_entidades_silver
    .filter(
        F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista"
        )
    )
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
    .withColumn(
        "sk_person_id",
        F.row_number()
        .over(Window.orderBy("nome_pessoa", "tipo_pessoa"))
        .cast("long"),
    )
    .select(
        "sk_person_id",
        "nome_pessoa",
        "tipo_pessoa",
    )
)

# As produtoras ficam em uma dimensão própria.
df_dim_companies = (
    df_entidades_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        F.col("nome_entidade").alias("nome_produtora")
    )
    .distinct()
    .withColumn(
        "sk_company_id",
        F.row_number()
        .over(Window.orderBy("nome_produtora"))
        .cast("long"),
    )
    .select("sk_company_id", "nome_produtora")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
dimensoes_cadastro = [
    (
        "dim_genres",
        df_dim_genres,
        "sk_genre_id",
        ["nome_genero"],
    ),
    (
        "dim_people",
        df_dim_people,
        "sk_person_id",
        ["nome_pessoa", "tipo_pessoa"],
    ),
    (
        "dim_companies",
        df_dim_companies,
        "sk_company_id",
        ["nome_produtora"],
    ),
]

resumo_dimensoes = []
problemas_dimensoes = []

for nome_tabela, df, chave, campos_cadastro in dimensoes_cadastro:
    total = df.count()
    chaves_distintas = df.select(chave).distinct().count()
    cadastros_distintos = (
        df.select(*campos_cadastro).distinct().count()
    )

    campo_ausente = F.lit(False)

    for campo in campos_cadastro:
        campo_ausente = (
            campo_ausente
            | F.col(campo).isNull()
            | (F.trim(F.col(campo)) == "")
        )

    registros_incompletos = df.filter(campo_ausente).count()

    if (
        total == 0
        or total != chaves_distintas
        or total != cadastros_distintos
        or registros_incompletos > 0
    ):
        problemas_dimensoes.append(nome_tabela)

    resumo_dimensoes.append((
        nome_tabela,
        total,
        chaves_distintas,
        cadastros_distintos,
        registros_incompletos,
    ))

display(
    spark.createDataFrame(
        resumo_dimensoes,
        [
            "dimensao",
            "total_registros",
            "chaves_distintas",
            "cadastros_distintos",
            "registros_incompletos",
        ],
    )
)

display(
    df_dim_people
    .groupBy("tipo_pessoa")
    .agg(F.count("*").alias("total_cadastros"))
    .orderBy("tipo_pessoa")
)

if problemas_dimensoes:
    raise ValueError(
        "Revise as dimensões: "
        + ", ".join(problemas_dimensoes)
    )

print("As três dimensões passaram pelas verificações de cadastro.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dimensao,total_registros,chaves_distintas,cadastros_distintos,registros_incompletos
dim_genres,19,19,19,0
dim_people,418060,418060,418060,0
dim_companies,44602,44602,44602,0


tipo_pessoa,total_cadastros
Ator,268936
Diretor,64134
Roteirista,84990


As três dimensões passaram pelas verificações de cadastro.


#### Resultado das dimensões de cadastro

Foram construídos 19 cadastros de gêneros, 418.060 cadastros de pessoas por tipo de atuação e 44.602 cadastros de produtoras, todos com chaves únicas e campos de identificação preenchidos. Os cadastros de pessoas estão distribuídos entre 268.936 atores, 64.134 diretores e 84.990 roteiristas. 

Essas quantidades representam combinações distintas de nome e tipo de atuação, não necessariamente pessoas diferentes: uma mesma pessoa pode aparecer em mais de uma função, e pessoas com mesmo nome podem ser agrupadas. Cada cadastro poderá ser associado a vários filmes pelas tabelas de relacionamento.

## Relacionamentos entre filmes, gêneros, pessoas e produtoras

Um filme pode ter vários gêneros, participantes e produtoras. Vamos representar essas associações em três tabelas de relacionamento, chamadas tabelas-ponte, contendo apenas as chaves das dimensões. Isso permite consultar essas associações sem multiplicar diretamente os registros da tabela fato. 

Utilizaremos os vínculos existentes na Silver e manteremos na Gold aqueles cujos filmes estão presentes em dim_movies. Os vínculos sem correspondência serão contabilizados nesta etapa e continuarão preservados na Silver. Também conferiremos se cada associação possui correspondência no cadastro de sua entidade.

In [0]:
# Identificadores originais e suas respectivas chaves Gold.
mapa_filmes = df_dim_movies.select(
    "id_filme",
    "sk_movie_id",
)

# Vínculos únicos recebidos da Silver.
vinculos_generos = (
    df_generos_silver
    .select("id_filme", "nome_genero")
    .distinct()
)

vinculos_pessoas = (
    df_entidades_silver
    .filter(
        F.col("tipo_entidade").isin(
            "Ator", "Diretor", "Roteirista"
        )
    )
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
)

vinculos_produtoras = (
    df_entidades_silver
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_produtora"),
    )
    .distinct()
)

# Mantemos apenas associações com filmes presentes na dimensão.
generos_com_filme = vinculos_generos.join(
    mapa_filmes,
    on="id_filme",
    how="inner",
)

pessoas_com_filme = vinculos_pessoas.join(
    mapa_filmes,
    on="id_filme",
    how="inner",
)

produtoras_com_filme = vinculos_produtoras.join(
    mapa_filmes,
    on="id_filme",
    how="inner",
)

# Substituímos os nomes pelas chaves dos cadastros.
# O left join permite detectar uma correspondência ausente,
# em vez de eliminar silenciosamente o vínculo.
df_bridge_movie_genre = (
    generos_com_filme
    .join(
        df_dim_genres,
        on="nome_genero",
        how="left",
    )
    .select("sk_movie_id", "sk_genre_id")
)

df_bridge_movie_person = (
    pessoas_com_filme
    .join(
        df_dim_people,
        on=["nome_pessoa", "tipo_pessoa"],
        how="left",
    )
    .select("sk_movie_id", "sk_person_id")
)

df_bridge_movie_company = (
    produtoras_com_filme
    .join(
        df_dim_companies,
        on="nome_produtora",
        how="left",
    )
    .select("sk_movie_id", "sk_company_id")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
configuracoes_pontes = [
    (
        "bridge_movie_genre",
        vinculos_generos,
        df_bridge_movie_genre,
        "sk_genre_id",
        df_dim_genres,
    ),
    (
        "bridge_movie_person",
        vinculos_pessoas,
        df_bridge_movie_person,
        "sk_person_id",
        df_dim_people,
    ),
    (
        "bridge_movie_company",
        vinculos_produtoras,
        df_bridge_movie_company,
        "sk_company_id",
        df_dim_companies,
    ),
]

resumo_pontes = []
problemas_pontes = []

for nome, origem, ponte, chave_entidade, dimensao in configuracoes_pontes:
    total_origem = origem.count()

    # Aqui contamos vínculos, não filmes distintos.
    sem_filme = (
        origem
        .join(
            mapa_filmes.select("id_filme"),
            on="id_filme",
            how="left_anti",
        )
        .count()
    )

    total_ponte = ponte.count()

    repeticoes = (
        total_ponte
        - ponte.select(
            "sk_movie_id", chave_entidade
        ).distinct().count()
    )

    chaves_ausentes = (
        ponte
        .filter(
            F.col("sk_movie_id").isNull()
            | F.col(chave_entidade).isNull()
        )
        .count()
    )

    filmes_sem_dimensao = (
        ponte
        .select("sk_movie_id")
        .distinct()
        .join(
            mapa_filmes.select("sk_movie_id"),
            on="sk_movie_id",
            how="left_anti",
        )
        .count()
    )

    entidades_sem_dimensao = (
        ponte
        .select(chave_entidade)
        .distinct()
        .join(
            dimensao.select(chave_entidade),
            on=chave_entidade,
            how="left_anti",
        )
        .count()
    )

    # Todo vínculo deve estar na ponte ou ter sido identificado
    # como sem correspondência em dim_movies.
    cobertura_confere = (
        total_origem == total_ponte + sem_filme
    )

    if (
        not cobertura_confere
        or repeticoes > 0
        or chaves_ausentes > 0
        or filmes_sem_dimensao > 0
        or entidades_sem_dimensao > 0
    ):
        problemas_pontes.append(nome)

    resumo_pontes.append((
        nome,
        total_origem,
        sem_filme,
        total_ponte,
        repeticoes,
        chaves_ausentes,
        filmes_sem_dimensao,
        entidades_sem_dimensao,
        cobertura_confere,
    ))

display(
    spark.createDataFrame(
        resumo_pontes,
        [
            "tabela",
            "vinculos_na_silver",
            "vinculos_sem_filme",
            "vinculos_na_gold",
            "repeticoes",
            "chaves_ausentes",
            "filmes_sem_dimensao",
            "entidades_sem_dimensao",
            "cobertura_confere",
        ],
    )
)

if problemas_pontes:
    raise ValueError(
        "Revise os relacionamentos: "
        + ", ".join(problemas_pontes)
    )

print(
    "As três tabelas de relacionamento passaram "
    "pelas verificações."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tabela,vinculos_na_silver,vinculos_sem_filme,vinculos_na_gold,repeticoes,chaves_ausentes,filmes_sem_dimensao,entidades_sem_dimensao,cobertura_confere
bridge_movie_genre,141356,1839,139517,0,0,0,0,true
bridge_movie_person,773131,10910,762221,0,0,0,0,true
bridge_movie_company,118521,1432,117089,0,0,0,0,true


As três tabelas de relacionamento passaram pelas verificações.


#### Resultado dos relacionamentos

Foram mantidos na Gold 139.517 vínculos entre filmes e gêneros, 762.221 entre filmes e pessoas e 117.089 entre filmes e produtoras. Não foram incluídos nas pontes 1.839 vínculos de gêneros, 10.910 de pessoas e 1.432 de produtoras porque seus filmes não possuem correspondência em dim_movies; esses registros continuam preservados na Silver. As quantidades representam associações, não filmes distintos. Não encontramos relacionamentos repetidos, chaves ausentes ou referências inexistentes nas dimensões. Em cada ponte, a soma dos vínculos mantidos com os vínculos sem filme corresponde à quantidade recebida da Silver.

## Dimensão de avaliações resumidas por filme

Conforme a atividade, dim_reviews terá uma linha por filme com avaliações disponíveis, contendo a quantidade de avaliações e a média das notas arredondada para duas casas decimais. A contagem incluirá avaliações sem nota, pois elas continuam sendo registros de participação dos usuários; a média considerará apenas as notas preenchidas. Se todas as notas de um filme estiverem ausentes, sua média permanecerá NULL. Avaliações de filmes sem correspondência em dim_movies serão contabilizadas separadamente e permanecerão na Silver. Filmes sem avaliações não receberão uma linha artificial nesta dimensão.

In [0]:
df_avaliacoes_silver = fontes_silver["tb_avaliacoes_usuarios"]

# Separamos as avaliações que possuem filme na dimensão.
avaliacoes_com_filme = (
    df_avaliacoes_silver
    .join(
        mapa_filmes,
        on="id_filme",
        how="inner",
    )
)

total_avaliacoes_origem = df_avaliacoes_silver.count()

avaliacoes_sem_filme = (
    df_avaliacoes_silver
    .join(
        mapa_filmes.select("id_filme"),
        on="id_filme",
        how="left_anti",
    )
    .count()
)

# count(*) inclui avaliações sem nota.
# avg ignora notas NULL e retorna NULL se não houver notas.
resumo_avaliacoes_filme = (
    avaliacoes_com_filme
    .groupBy("sk_movie_id")
    .agg(
        F.count("*").alias("quantidade_avaliacoes"),
        F.round(
            F.avg("nota_usuario"), 2
        ).cast("double").alias("nota_media_usuarios"),
    )
)

# A atividade exige INT para a contagem.
if (
    resumo_avaliacoes_filme
    .filter(F.col("quantidade_avaliacoes") > 2147483647)
    .limit(1)
    .count()
):
    raise ValueError(
        "Existe uma contagem de avaliações acima do limite de INT."
    )

df_dim_reviews = (
    resumo_avaliacoes_filme
    .withColumn(
        "sk_review_id",
        F.row_number()
        .over(Window.orderBy("sk_movie_id"))
        .cast("long"),
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        F.col("quantidade_avaliacoes")
        .cast("int")
        .alias("qtd_avaliacoes_usuarios"),
        "nota_media_usuarios",
    )
)

conferencia_avaliacoes = df_dim_reviews.agg(
    F.count("*").alias("filmes_com_avaliacoes"),
    F.countDistinct("sk_review_id").alias("chaves_distintas"),
    F.countDistinct("sk_movie_id").alias("filmes_distintos"),
    F.coalesce(
        F.sum("qtd_avaliacoes_usuarios"),
        F.lit(0),
    ).alias("avaliacoes_representadas"),
    F.count(
        F.when(F.col("nota_media_usuarios").isNull(), 1)
    ).alias("filmes_sem_nota_media"),
).first()

avaliacoes_representadas = (
    conferencia_avaliacoes["avaliacoes_representadas"]
)

if (
    total_avaliacoes_origem
    != avaliacoes_representadas + avaliacoes_sem_filme
):
    raise ValueError(
        "A quantidade de avaliações não foi preservada na agregação."
    )

if not (
    conferencia_avaliacoes["filmes_com_avaliacoes"]
    == conferencia_avaliacoes["chaves_distintas"]
    == conferencia_avaliacoes["filmes_distintos"]
):
    raise ValueError(
        "A dimensão de avaliações possui chaves ou filmes repetidos."
    )

medias_invalidas = (
    df_dim_reviews
    .filter(
        F.col("nota_media_usuarios").isNotNull()
        & ~F.col("nota_media_usuarios").between(0, 10)
    )
    .limit(1)
    .count()
)

if medias_invalidas:
    raise ValueError("Existe uma média fora da faixa de 0 a 10.")

display(
    spark.createDataFrame(
        [(
            total_avaliacoes_origem,
            avaliacoes_sem_filme,
            avaliacoes_representadas,
            conferencia_avaliacoes["filmes_com_avaliacoes"],
            conferencia_avaliacoes["filmes_sem_nota_media"],
        )],
        [
            "avaliacoes_na_silver",
            "avaliacoes_sem_filme",
            "avaliacoes_representadas_na_gold",
            "filmes_com_avaliacoes",
            "filmes_sem_nota_media",
        ],
    )
)

display(
    df_dim_reviews
    .join(
        df_dim_movies.select("sk_movie_id", "titulo"),
        on="sk_movie_id",
        how="inner",
    )
    .select(
        "sk_review_id",
        "sk_movie_id",
        "titulo",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios",
    )
    .orderBy(
        F.desc("qtd_avaliacoes_usuarios"),
        "sk_movie_id",
    )
    .limit(10)
)

print("A dimensão de avaliações passou pelas verificações.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


avaliacoes_na_silver,avaliacoes_sem_filme,avaliacoes_representadas_na_gold,filmes_com_avaliacoes,filmes_sem_nota_media
32412,505,31907,27227,1190


sk_review_id,sk_movie_id,titulo,qtd_avaliacoes_usuarios,nota_media_usuarios
4204,14754,Laddoo,5,4.73
512,1832,Jezioro Słone,4,5.28
1463,5279,Space Chase USA,4,4.38
1515,5454,Dikkat Köpek Var,4,3.35
2062,7364,Zo doo wiejleu dat: een eeuw Twente op film,4,6.23
2369,8437,Wolfswinkel,4,3.93
4909,17298,Dabba,4,4.18
5246,18502,City of Gold,4,4.9
5863,20670,wwe roadblock 2016,4,7.4
5935,20900,Ernestine & Kit,4,7.07


A dimensão de avaliações passou pelas verificações.


#### Resultado do resumo das avaliações

A dimensão resume 31.907 avaliações distribuídas entre 27.227 filmes. Outras 505 avaliações não possuem filme correspondente em dim_movies e permanecem preservadas na Silver. Todas as avaliações de origem foram contabilizadas entre esses dois grupos. Dos filmes representados, 1.190 possuem avaliações, mas nenhuma nota preenchida; por isso, sua média permaneceu NULL. As verificações de unicidade passaram, e as médias preenchidas estão entre 0 e 10.

## Tabela fato de desempenho dos filmes

Vamos reunir as métricas financeiras e de engajamento dos filmes com status 'Lançado', mantendo exatamente uma linha por filme. Utilizaremos junções à esquerda para preservar esses filmes mesmo quando não houver registro financeiro ou de engajamento correspondente. Valores ausentes continuarão como NULL, pois ausência de informação não significa zero. 

As tabelas de gêneros, pessoas e produtoras não participarão diretamente desta montagem, evitando multiplicar métricas pelos vários relacionamentos de um filme. Antes das junções, verificaremos a unicidade dos identificadores nas duas fontes de métricas.

In [0]:
df_financeiro_silver = fontes_silver["tb_financeiro_filmes"]
df_metricas_silver = fontes_silver["tb_metricas_engajamento"]

# As duas fontes devem possuir, no máximo, uma linha por filme.
for nome, df in [
    ("tb_financeiro_filmes", df_financeiro_silver),
    ("tb_metricas_engajamento", df_metricas_silver),
]:
    possui_id_invalido = (
        df.filter(
            F.col("id_filme").isNull()
            | (F.trim("id_filme") == "")
        )
        .limit(1)
        .count()
    )

    possui_repeticao = (
        df.groupBy("id_filme")
        .count()
        .filter(F.col("count") > 1)
        .limit(1)
        .count()
    )

    if possui_id_invalido or possui_repeticao:
        raise ValueError(
            f"Revise a identificação dos filmes em {nome}."
        )

filmes_lancados = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme")
)

colunas_financeiras = [
    "orcamento_usd",
    "receita_usd",
    "lucro_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_brl",
]

colunas_engajamento = [
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb",
]

# Os indicadores distinguem a ausência de uma linha na fonte
# da existência de uma linha com métricas NULL.
financeiro_para_fato = (
    df_financeiro_silver
    .select("id_filme", *colunas_financeiras)
    .withColumn("possui_registro_financeiro", F.lit(True))
)

metricas_para_fato = (
    df_metricas_silver
    .select("id_filme", *colunas_engajamento)
    .withColumn("possui_registro_engajamento", F.lit(True))
)

df_fato_preparacao = (
    filmes_lancados
    .join(
        financeiro_para_fato,
        on="id_filme",
        how="left",
    )
    .join(
        metricas_para_fato,
        on="id_filme",
        how="left",
    )
)

# Selecionamos as colunas exigidas para a tabela fato.
# Os tipos numéricos já foram definidos na Silver.
df_fact_movies_performance = (
    df_fato_preparacao
    .select(
        "sk_movie_id",
        *colunas_financeiras,
        *colunas_engajamento,
    )
)

total_lancados = filmes_lancados.count()

resumo_fato = df_fato_preparacao.agg(
    F.count("*").alias("registros_na_fato"),
    F.countDistinct("sk_movie_id").alias("filmes_distintos"),
    F.count(
        F.when(
            F.col("possui_registro_financeiro").isNull(), 1
        )
    ).alias("filmes_sem_registro_financeiro"),
    F.count(
        F.when(
            F.col("possui_registro_engajamento").isNull(), 1
        )
    ).alias("filmes_sem_registro_engajamento"),
).first()

if not (
    total_lancados
    == resumo_fato["registros_na_fato"]
    == resumo_fato["filmes_distintos"]
):
    raise ValueError(
        "A montagem da fato alterou a quantidade de filmes "
        "ou gerou registros repetidos."
    )

# Conferimos os tipos exigidos pelo documento.
tipos_esperados_fato = {
    "sk_movie_id": "bigint",
    **{
        coluna: "decimal(18,2)"
        for coluna in colunas_financeiras
    },
    "popularidade": "double",
    "nota_media_tmdb": "double",
    "qtd_votos_tmdb": "int",
    "nota_media_imdb": "double",
    "qtd_votos_imdb": "int",
}

tipos_encontrados = dict(df_fact_movies_performance.dtypes)

if tipos_encontrados != tipos_esperados_fato:
    raise ValueError(
        "Os tipos da tabela fato diferem dos exigidos. "
        f"Tipos encontrados: {tipos_encontrados}"
    )

display(
    spark.createDataFrame(
        [(
            total_lancados,
            resumo_fato["registros_na_fato"],
            resumo_fato["filmes_distintos"],
            resumo_fato["filmes_sem_registro_financeiro"],
            resumo_fato["filmes_sem_registro_engajamento"],
        )],
        [
            "filmes_lancados_na_dimensao",
            "registros_na_fato",
            "filmes_distintos",
            "filmes_sem_registro_financeiro",
            "filmes_sem_registro_engajamento",
        ],
    )
)

display(
    df_fact_movies_performance
    .join(
        df_dim_movies.select("sk_movie_id", "titulo"),
        on="sk_movie_id",
        how="inner",
    )
    .select(
        "sk_movie_id",
        "titulo",
        "orcamento_usd",
        "receita_usd",
        "lucro_usd",
        "popularidade",
        "nota_media_tmdb",
    )
    .orderBy("sk_movie_id")
    .limit(10)
)

print(
    "A tabela fato preservou uma linha por filme lançado "
    "e apresentou os tipos exigidos."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


filmes_lancados_na_dimensao,registros_na_fato,filmes_distintos,filmes_sem_registro_financeiro,filmes_sem_registro_engajamento
96241,96241,96241,137,922


sk_movie_id,titulo,orcamento_usd,receita_usd,lucro_usd,popularidade,nota_media_tmdb
1,Purple Beatz,null,null,null,1.132,0.0
2,Aisha Brown: The First Black Woman Ever,null,null,null,0.6,0.0
3,KYLE BROWNRIGG: INTRODUCING LYLE,null,null,null,0.6,0.0
4,Worth Your Weight in Gold,null,null,null,1.169,0.0
5,On va manquer !,null,null,null,0.6,0.0
6,58 Hours: The Baby Jessica Story,null,null,null,0.615,0.0
7,One Hundred Years and Hope,null,null,null,0.6,0.0
8,Homecoming,4700000.00,null,null,1.489,6.75
9,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,null,null,null,0.6,0.0
10,A Chance To Win,6000000.00,null,null,13.212,6.8


A tabela fato preservou uma linha por filme lançado e apresentou os tipos exigidos.


#### Resultado da tabela fato

A tabela fato contém 96.241 registros, correspondentes aos 96.241 filmes com status Lançado na dimensão, sem repetição de filmes após as junções. Foram preservados 137 filmes sem registro financeiro correspondente e 922 sem registro de engajamento. 

Essas contagens indicam ausência de correspondência nas fontes; registros encontrados também podem conter métricas ausentes. Os valores NULL foram mantidos sem substituição por zero, e os tipos das colunas passaram pela conferência exigida para a fato.

## Preparação do contexto para IA

Vamos construir gold_genai_movies_context com as três colunas solicitadas: movie_id, title e llm_context_document. 

O documento será um texto corrido com título, ano, receita, orçamento, atores, diretores e sinopse. Usaremos todos os filmes de dim_movies e uma junção à esquerda com a fato, preservando também aqueles sem métricas financeiras. Atores e diretores serão recuperados de dim_people pela bridge_movie_person e agrupados antes da junção, evitando multiplicar filmes. 

Como não preservamos uma ordem de destaque do elenco, os atores serão apresentados em ordem alfabética, sem afirmar que essa ordem representa protagonismo. Antes da concatenação, vamos contar as ausências e definir textos substitutos para que um campo NULL não torne o documento inteiro nulo. Os valores financeiros serão apresentados em dólares.

In [0]:
# Recuperamos os nomes das pessoas relacionadas a cada filme.
pessoas_por_filme = (
    df_bridge_movie_person
    .join(
        df_dim_people,
        on="sk_person_id",
        how="inner",
    )
)

# Agregamos antes de juntar à dimensão de filmes.
# A ordenação mantém o texto reproduzível entre execuções.
elenco_direcao_por_filme = (
    pessoas_por_filme
    .groupBy("sk_movie_id")
    .agg(
        F.sort_array(
            F.collect_set(
                F.when(
                    F.col("tipo_pessoa") == "Ator",
                    F.col("nome_pessoa"),
                )
            )
        ).alias("atores"),
        F.sort_array(
            F.collect_set(
                F.when(
                    F.col("tipo_pessoa") == "Diretor",
                    F.col("nome_pessoa"),
                )
            )
        ).alias("diretores"),
    )
    .select(
        "sk_movie_id",
        F.concat_ws(", ", "atores").alias("atores_texto"),
        F.concat_ws(", ", "diretores").alias("diretores_texto"),
    )
)

df_contexto_base = (
    df_dim_movies
    .join(
        df_fact_movies_performance.select(
            "sk_movie_id",
            "receita_usd",
            "orcamento_usd",
        ),
        on="sk_movie_id",
        how="left",
    )
    .join(
        elenco_direcao_por_filme,
        on="sk_movie_id",
        how="left",
    )
)

# Para textos, consideramos ausentes NULL e espaços em branco.
def texto_ausente(coluna):
    return (
        F.col(coluna).isNull()
        | F.col(coluna).rlike(r"^\s*$")
    )

condicoes_ausencia = {
    "titulo": texto_ausente("titulo"),
    "ano_lancamento": F.col("ano_lancamento").isNull(),
    "receita_usd": F.col("receita_usd").isNull(),
    "orcamento_usd": F.col("orcamento_usd").isNull(),
    "atores_texto": texto_ausente("atores_texto"),
    "diretores_texto": texto_ausente("diretores_texto"),
    "sinopse": texto_ausente("sinopse"),
}

ausencias_contexto = df_contexto_base.agg(
    *[
        F.count(F.when(condicao, 1)).alias(campo)
        for campo, condicao in condicoes_ausencia.items()
    ]
).first()

display(
    spark.createDataFrame(
        [
            (campo, int(ausencias_contexto[campo]))
            for campo in condicoes_ausencia
        ],
        ["campo", "filmes_com_informacao_ausente"],
    )
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


campo,filmes_com_informacao_ausente
titulo,0
ano_lancamento,2
receita_usd,94331
orcamento_usd,89631
atores_texto,18030
diretores_texto,12726
sinopse,14087


In [0]:
# Preparamos cada campo antes da concatenação.
def texto_com_fallback(coluna, alternativa):
    return F.when(
        texto_ausente(coluna),
        F.lit(alternativa),
    ).otherwise(F.trim(F.col(coluna)))


titulo_contexto = texto_com_fallback(
    "titulo", "Título não informado"
)

ano_contexto = F.coalesce(
    F.col("ano_lancamento").cast("string"),
    F.lit("ano não informado"),
)

# A indicação de USD torna explícita a moeda utilizada.
receita_contexto = F.coalesce(
    F.concat(
        F.lit("USD "),
        F.col("receita_usd").cast("string"),
    ),
    F.lit("valor não informado"),
)

orcamento_contexto = F.coalesce(
    F.concat(
        F.lit("USD "),
        F.col("orcamento_usd").cast("string"),
    ),
    F.lit("valor não informado"),
)

atores_contexto = texto_com_fallback(
    "atores_texto", "elenco não informado"
)

diretores_contexto = texto_com_fallback(
    "diretores_texto", "direção não informada"
)

sinopse_contexto = texto_com_fallback(
    "sinopse", "Sinopse não informada"
)

df_gold_genai_movies_context = (
    df_contexto_base
    .select(
        F.col("id_filme").alias("movie_id"),
        titulo_contexto.alias("title"),
        F.concat(
            F.lit("O filme "),
            titulo_contexto,
            F.lit(", com ano de lançamento "),
            ano_contexto,
            F.lit(", tem receita registrada de "),
            receita_contexto,
            F.lit(" e orçamento registrado de "),
            orcamento_contexto,
            F.lit(". Seu elenco reúne "),
            atores_contexto,
            F.lit(", e sua direção é atribuída a "),
            diretores_contexto,
            F.lit(". A sinopse disponível é: "),
            sinopse_contexto,
        ).alias("llm_context_document"),
    )
)

resumo_contexto = df_gold_genai_movies_context.agg(
    F.count("*").alias("total_documentos"),
    F.countDistinct("movie_id").alias("filmes_distintos"),
    F.count(
        F.when(
            F.col("llm_context_document").isNull()
            | F.col("llm_context_document").rlike(r"^\s*$"),
            1,
        )
    ).alias("documentos_ausentes"),
).first()

total_filmes_dimensao = df_dim_movies.count()

if not (
    resumo_contexto["total_documentos"]
    == resumo_contexto["filmes_distintos"]
    == total_filmes_dimensao
):
    raise ValueError(
        "A construção do contexto perdeu ou multiplicou filmes."
    )

if resumo_contexto["documentos_ausentes"] > 0:
    raise ValueError(
        "Existem documentos de contexto nulos ou vazios."
    )

display(
    spark.createDataFrame(
        [(
            total_filmes_dimensao,
            resumo_contexto["total_documentos"],
            resumo_contexto["filmes_distintos"],
            resumo_contexto["documentos_ausentes"],
        )],
        [
            "filmes_na_dimensao",
            "documentos_gerados",
            "filmes_distintos",
            "documentos_ausentes",
        ],
    )
)

# Uma amostra com informações financeiras ausentes
# ajuda a conferir o funcionamento dos textos substitutos.
ids_para_amostra = (
    df_contexto_base
    .filter(
        F.col("receita_usd").isNull()
        | F.col("orcamento_usd").isNull()
    )
    .select(F.col("id_filme").alias("movie_id"))
    .orderBy("movie_id")
    .limit(3)
)

display(
    df_gold_genai_movies_context
    .join(ids_para_amostra, on="movie_id", how="inner")
    .orderBy("movie_id")
)

print(
    "Foi gerado um documento por filme, "
    "sem documentos nulos ou vazios."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


filmes_na_dimensao,documentos_gerados,filmes_distintos,documentos_ausentes
97594,97594,97594,0


movie_id,title,llm_context_document
1000004,Purple Beatz,"O filme Purple Beatz, com ano de lançamento 2022, tem receita registrada de valor não informado e orçamento registrado de valor não informado. Seu elenco reúne Aron Von Andrian, Erika Alexander, Izzy Jones, Steven Michael-o’hara, Tedroy Newell, e sua direção é atribuída a Lola Atkins. A sinopse disponível é: Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry."
1000005,Aisha Brown: The First Black Woman Ever,"O filme Aisha Brown: The First Black Woman Ever, com ano de lançamento 2020, tem receita registrada de valor não informado e orçamento registrado de valor não informado. Seu elenco reúne Aisha Brown, e sua direção é atribuída a Mathieu Baer. A sinopse disponível é: Sinopse não informada"
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,"O filme KYLE BROWNRIGG: INTRODUCING LYLE, com ano de lançamento 2022, tem receita registrada de valor não informado e orçamento registrado de valor não informado. Seu elenco reúne Kyle Brownrigg, e sua direção é atribuída a Mathieu Baer. A sinopse disponível é: Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle."


Foi gerado um documento por filme, sem documentos nulos ou vazios.


#### Resultado da preparação do contexto para IA

Foram gerados 97.594 documentos, um por filme da dimensão, sem documentos nulos ou vazios. Na base utilizada para a composição, encontramos dois filmes sem ano de lançamento, 94.331 sem receita, 89.631 sem orçamento, 18.030 sem elenco, 12.726 sem direção e 14.087 sem sinopse. 

Essas ausências podem ocorrer no mesmo filme e não devem ser somadas como filmes distintos. As ausências financeiras também incluem filmes fora do recorte da tabela fato, que contém somente os lançados. A amostra confirmou que os textos substitutos mantêm o documento legível sem inventar informações. As sinopses foram preservadas no idioma disponível na origem, e o elenco foi apresentado em ordem alfabética, sem atribuição de protagonismo.

## Validação e gravação da camada Gold

Vamos conferir as dez tabelas antes de gravá-las: suas chaves, os relacionamentos entre dimensões e pontes, o conjunto de filmes da fato e a cobertura dos documentos de contexto. Depois, salvaremos as tabelas em Delta, substituindo a versão anterior para evitar acúmulo em reexecuções. 

Como as chaves foram geradas por ordenação, todas as tabelas relacionadas devem ser reconstruídas na mesma execução. A publicação ocorre separadamente por tabela; se houver uma interrupção, a Gold só deverá ser consumida após a conclusão de uma nova execução e de sua conferência.

In [0]:
validacao_gold_aprovada = False

tabelas_gold = {
    "dim_movies": df_dim_movies,
    "dim_genres": df_dim_genres,
    "dim_people": df_dim_people,
    "dim_companies": df_dim_companies,
    "dim_reviews": df_dim_reviews,
    "bridge_movie_genre": df_bridge_movie_genre,
    "bridge_movie_person": df_bridge_movie_person,
    "bridge_movie_company": df_bridge_movie_company,
    "fact_movies_performance": df_fact_movies_performance,
    "gold_genai_movies_context": df_gold_genai_movies_context,
}

chaves_gold = {
    "dim_movies": ["sk_movie_id"],
    "dim_genres": ["sk_genre_id"],
    "dim_people": ["sk_person_id"],
    "dim_companies": ["sk_company_id"],
    "dim_reviews": ["sk_review_id"],
    "bridge_movie_genre": ["sk_movie_id", "sk_genre_id"],
    "bridge_movie_person": ["sk_movie_id", "sk_person_id"],
    "bridge_movie_company": ["sk_movie_id", "sk_company_id"],
    "fact_movies_performance": ["sk_movie_id"],
    "gold_genai_movies_context": ["movie_id"],
}

resumo_validacao_gold = []
problemas_gold = []

for nome, df in tabelas_gold.items():
    chaves = chaves_gold[nome]
    total = df.count()
    repeticoes = total - df.select(*chaves).distinct().count()

    chave_ausente = F.lit(False)

    for chave in chaves:
        chave_ausente = (
            chave_ausente
            | F.col(chave).isNull()
            | (F.trim(F.col(chave).cast("string")) == "")
        )

    ausentes = df.filter(chave_ausente).count()

    if total == 0 or repeticoes > 0 or ausentes > 0:
        problemas_gold.append(
            f"{nome}: tabela vazia, chave repetida ou ausente."
        )

    resumo_validacao_gold.append((
        nome, total, repeticoes, ausentes
    ))

# Conferimos todas as referências entre as tabelas.
relacionamentos_gold = [
    ("dim_reviews", "sk_movie_id", "dim_movies"),
    ("fact_movies_performance", "sk_movie_id", "dim_movies"),
    ("bridge_movie_genre", "sk_movie_id", "dim_movies"),
    ("bridge_movie_genre", "sk_genre_id", "dim_genres"),
    ("bridge_movie_person", "sk_movie_id", "dim_movies"),
    ("bridge_movie_person", "sk_person_id", "dim_people"),
    ("bridge_movie_company", "sk_movie_id", "dim_movies"),
    ("bridge_movie_company", "sk_company_id", "dim_companies"),
]

resumo_relacionamentos = []

for origem, chave, destino in relacionamentos_gold:
    referencias_invalidas = (
        tabelas_gold[origem]
        .select(chave)
        .distinct()
        .join(
            tabelas_gold[destino].select(chave),
            on=chave,
            how="left_anti",
        )
        .count()
    )

    if referencias_invalidas > 0:
        problemas_gold.append(
            f"{origem}: referência inválida em {chave}."
        )

    resumo_relacionamentos.append((
        origem, chave, destino, referencias_invalidas
    ))

# Comparamos conjuntos, não apenas quantidades.
def mesmo_conjunto(df_a, df_b):
    return (
        df_a.exceptAll(df_b).limit(1).count() == 0
        and df_b.exceptAll(df_a).limit(1).count() == 0
    )


filmes_esperados_fato = (
    df_dim_movies
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id")
)

if not mesmo_conjunto(
    filmes_esperados_fato,
    df_fact_movies_performance.select("sk_movie_id"),
):
    problemas_gold.append(
        "A fato não corresponde ao conjunto de filmes lançados."
    )

if not mesmo_conjunto(
    df_dim_movies.select(F.col("id_filme").alias("movie_id")),
    df_gold_genai_movies_context.select("movie_id"),
):
    problemas_gold.append(
        "O contexto não corresponde ao conjunto de filmes da dimensão."
    )

if (
    df_dim_reviews.groupBy("sk_movie_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
):
    problemas_gold.append(
        "Existe mais de um resumo de avaliações por filme."
    )

if (
    df_gold_genai_movies_context
    .filter(
        F.col("llm_context_document").isNull()
        | F.col("llm_context_document").rlike(r"^\s*$")
    )
    .limit(1)
    .count()
):
    problemas_gold.append("Existe documento de contexto vazio.")

display(
    spark.createDataFrame(
        resumo_validacao_gold,
        [
            "tabela",
            "total_registros",
            "repeticoes_da_chave",
            "chaves_ausentes",
        ],
    )
)

display(
    spark.createDataFrame(
        resumo_relacionamentos,
        [
            "tabela_origem",
            "chave",
            "dimensao_destino",
            "referencias_invalidas",
        ],
    )
)

if problemas_gold:
    raise ValueError(" | ".join(problemas_gold))

validacao_gold_aprovada = True
print("As dez tabelas passaram pelas verificações finais.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tabela,total_registros,repeticoes_da_chave,chaves_ausentes
dim_movies,97594,0,0
dim_genres,19,0,0
dim_people,418060,0,0
dim_companies,44602,0,0
dim_reviews,27227,0,0
bridge_movie_genre,139517,0,0
bridge_movie_person,762221,0,0
bridge_movie_company,117089,0,0
fact_movies_performance,96241,0,0
gold_genai_movies_context,97594,0,0


tabela_origem,chave,dimensao_destino,referencias_invalidas
dim_reviews,sk_movie_id,dim_movies,0
fact_movies_performance,sk_movie_id,dim_movies,0
bridge_movie_genre,sk_movie_id,dim_movies,0
bridge_movie_genre,sk_genre_id,dim_genres,0
bridge_movie_person,sk_movie_id,dim_movies,0
bridge_movie_person,sk_person_id,dim_people,0
bridge_movie_company,sk_movie_id,dim_movies,0
bridge_movie_company,sk_company_id,dim_companies,0


As dez tabelas passaram pelas verificações finais.


In [0]:
if not globals().get("validacao_gold_aprovada", False):
    raise ValueError(
        "Execute a validação da célula 26 antes de gravar."
    )

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {catalogo}.{schema_gold}"
)

# Reconstruímos o conjunto completo de tabelas.
for nome, df in tabelas_gold.items():
    destino = f"{catalogo}.{schema_gold}.{nome}"

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(destino)
    )

# Conferimos o conteúdo efetivamente salvo.
resumo_gravacao_gold = []
falhas_gravacao_gold = []

for nome, df_preparado in tabelas_gold.items():
    destino = f"{catalogo}.{schema_gold}.{nome}"
    df_salvo = spark.table(destino)

    if df_preparado.dtypes != df_salvo.dtypes:
        raise ValueError(
            f"A estrutura salva difere da preparada: {destino}"
        )

    total_preparado = df_preparado.count()
    total_salvo = df_salvo.count()

    conteudo_confere = mesmo_conjunto(df_preparado, df_salvo)

    formato = (
        spark.sql(f"DESCRIBE DETAIL {destino}")
        .select("format")
        .first()["format"]
    )

    if not conteudo_confere or formato.lower() != "delta":
        falhas_gravacao_gold.append(destino)

    resumo_gravacao_gold.append((
        destino,
        total_preparado,
        total_salvo,
        formato,
        conteudo_confere,
    ))

display(
    spark.createDataFrame(
        resumo_gravacao_gold,
        [
            "tabela",
            "registros_preparados",
            "registros_salvos",
            "formato",
            "conteudo_confere",
        ],
    )
)

if falhas_gravacao_gold:
    raise ValueError(
        "Falha na conferência: "
        + ", ".join(falhas_gravacao_gold)
    )

print(
    "As dez tabelas Gold foram gravadas em Delta "
    "e conferidas com os dados preparados."
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


tabela,registros_preparados,registros_salvos,formato,conteudo_confere
workspace.gold.dim_movies,97594,97594,delta,true
workspace.gold.dim_genres,19,19,delta,true
workspace.gold.dim_people,418060,418060,delta,true
workspace.gold.dim_companies,44602,44602,delta,true
workspace.gold.dim_reviews,27227,27227,delta,true
workspace.gold.bridge_movie_genre,139517,139517,delta,true
workspace.gold.bridge_movie_person,762221,762221,delta,true
workspace.gold.bridge_movie_company,117089,117089,delta,true
workspace.gold.fact_movies_performance,96241,96241,delta,true
workspace.gold.gold_genai_movies_context,97594,97594,delta,true


As dez tabelas Gold foram gravadas em Delta e conferidas com os dados preparados.


#### Resultado da validação e gravação da Gold

As dez tabelas passaram pelas verificações de unicidade e preenchimento das chaves, sem referências inválidas nos relacionamentos conferidos. A tabela fato preservou os 96.241 filmes lançados, e a tabela de contexto manteve um documento para cada um dos 97.594 filmes da dimensão. 

Todas as tabelas foram gravadas em Delta no schema workspace.gold, e a comparação confirmou a correspondência integral entre os dados preparados e os dados salvos. Essa conferência valida a gravação e a estrutura dos relacionamentos, mantendo as limitações de qualidade e cobertura documentadas nas etapas anteriores.

## Perguntas da atividade:

#### 1. Qual é a receita total em reais somada de todos os filmes da base?

A receita total disponível é de R$ 836.759.877.340,86, considerando os filmes lançados representados na tabela fato. Dos 96.241 filmes, apenas 3.263 possuem receita informada, enquanto 92.978 permanecem sem esse valor.

Portanto, a soma representa as receitas conhecidas, sem estimar valores ausentes. A conversão utiliza a taxa de câmbio aplicada na Silver, não a cotação histórica do lançamento de cada filme.

In [0]:
# 1. Qual é a receita total em reais?

fato_gold = spark.table(
    f"{catalogo}.{schema_gold}.fact_movies_performance"
)

filmes_gold = spark.table(
    f"{catalogo}.{schema_gold}.dim_movies"
)

df_receita_total = fato_gold.agg(
    F.sum("receita_brl").alias("receita_total_brl"),
    F.count("*").alias("filmes_na_fato"),
    F.count("receita_brl").alias("filmes_com_receita_informada"),
    F.count(
        F.when(F.col("receita_brl").isNull(), 1)
    ).alias("filmes_sem_receita_informada"),
)

display(df_receita_total)

receita_total_brl,filmes_na_fato,filmes_com_receita_informada,filmes_sem_receita_informada
836759877340.86,96241,3263,92978


#### 2. Quais são os cinco filmes com maior popularidade?

Os cinco maiores valores de popularidade registrados são blue beetle (2994.357), Gran Turismo (2680.593), The Fear Footage 2: Curse of the Tape (2019), Battipaglia 1969 (1969) e The Nun II (1692.778). 

Os valores 2019 e 1969 chamam atenção por se parecerem com anos, considerando os deslocamentos de colunas identificados na origem. Isso é um indício para revisão, não uma confirmação de erro. Como são números positivos, passaram pelas regras numéricas implementadas; por isso, o ranking deve ser interpretado com essa ressalva de qualidade.

In [0]:
# 2. Quais são os cinco filmes com maior popularidade?

df_top5_popularidade = (
    fato_gold
    .filter(F.col("popularidade").isNotNull())
    .join(
        filmes_gold.select("sk_movie_id", "id_filme", "titulo"),
        on="sk_movie_id",
        how="inner",
    )
    .orderBy(
        F.desc("popularidade"),
        F.asc("id_filme"),
    )
    .limit(5)
    .select("titulo", "popularidade")
)

display(df_top5_popularidade)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Fear Footage 2: Curse of the Tape,2019.0
Battipaglia 1969,1969.0
The Nun II,1692.778


#### 3. Quantos filmes cada gênero possui?

Drama apresenta a maior quantidade, com 32.062 filmes, seguido por Documentary, com 18.848, e Comedy, com 18.482. A tabela apresenta os 19 gêneros em ordem decrescente, chegando a Western, com 405 filmes. A contagem considera os filmes da dimensão com associação válida a cada gênero, incluindo diferentes status de produção. Como um filme pode pertencer a várias categorias, a soma dessas quantidades não representa o total de filmes distintos da base.

In [0]:
# 3. Quantos filmes cada gênero possui?

generos_gold = spark.table(
    f"{catalogo}.{schema_gold}.dim_genres"
)

ponte_generos_gold = spark.table(
    f"{catalogo}.{schema_gold}.bridge_movie_genre"
)

contagem_generos = (
    ponte_generos_gold
    .groupBy("sk_genre_id")
    .agg(
        F.countDistinct("sk_movie_id").alias("quantidade_filmes")
    )
)

df_filmes_por_genero = (
    generos_gold
    .join(
        contagem_generos,
        on="sk_genre_id",
        how="left",
    )
    .select(
        "nome_genero",
        F.coalesce(
            F.col("quantidade_filmes"),
            F.lit(0).cast("long"),
        ).alias("quantidade_filmes"),
    )
    .orderBy(
        F.desc("quantidade_filmes"),
        F.asc("nome_genero"),
    )
)

display(df_filmes_por_genero)

nome_genero,quantidade_filmes
Drama,32062
Documentary,18848
Comedy,18482
Thriller,10231
Horror,9654
Romance,7596
Action,6019
Crime,4713
Animation,4438
TV Movie,4061


#### 4. Quais são os dez filmes de maior receita e suas posições no ranking?

Avengers: Endgame lidera o ranking com US$ 2.800.000.000,00, equivalentes a R$ 14.439.320.000,00, seguido por Avatar: The Way of Water e AVENGERS: INFINITY WAR. Completam os dez primeiros spider-man: no way home, The Lion King, Top Gun: Maverick, Barbie, The Super Mario Bros. Movie, Black Panther e Star Wars: The Last Jedi. A tabela apresenta as receitas em dólares e reais para cada título. A função RANK atribuiu posições de 1 a 10, sem empates entre os valores exibidos.

In [0]:
# 4. Quais são os dez filmes de maior receita?

janela_ranking_receita = Window.orderBy(
    F.desc("receita_usd")
)

df_top10_receita = (
    fato_gold
    .filter(F.col("receita_usd").isNotNull())
    .join(
        filmes_gold.select("sk_movie_id", "id_filme", "titulo"),
        on="sk_movie_id",
        how="inner",
    )
    .withColumn(
        "posicao_ranking",
        F.rank().over(janela_ranking_receita),
    )
    .orderBy(
        F.desc("receita_usd"),
        F.asc("id_filme"),
    )
    .limit(10)
    .select(
        "posicao_ranking",
        "titulo",
        "receita_usd",
        "receita_brl",
    )
)

display(df_top10_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2800000000.00,14439320000.00
2,Avatar: The Way of Water,2320250281.00,11965298674.09
3,AVENGERS: INFINITY WAR,2052415039.00,10584099114.62
4,spider-man: no way home,1921847111.00,9910773366.72
5,The Lion King,1663075401.00,8576313535.42
6,Top Gun: Maverick,1488732821.00,7677246284.61
7,Barbie,1428545028.00,7366863854.89
8,The Super Mario Bros. Movie,1355725263.00,6991339608.76
9,Black Panther,1349926083.00,6961433817.42
10,Star Wars: The Last Jedi,1332698830.00,6872594596.43


#### 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos dois anos?

Kevin Hart liderou com 64 participações em filmes distintos, considerando lançamentos posteriores a 19/02/2024 e até 19/02/2026, inclusive. O limite superior corresponde ao lançamento realizado mais recente identificado na base, após excluir datas futuras e filmes sem status Lançado. Não houve empate na liderança. O resultado considera os vínculos disponíveis na Gold e a identificação dos atores por nome, mantendo as limitações de qualidade documentadas nas etapas anteriores.

In [0]:
# 5. Qual ator teve mais participações nos últimos dois anos?

pessoas_gold = spark.table(
    f"{catalogo}.{schema_gold}.dim_people"
)

ponte_pessoas_gold = spark.table(
    f"{catalogo}.{schema_gold}.bridge_movie_person"
)

# O limite é o lançamento realizado mais recente da base.
filmes_lancados_validos = filmes_gold.filter(
    (F.col("status_filme") == "Lançado")
    & F.col("data_lancamento").isNotNull()
    & (F.col("data_lancamento") <= F.lit(data_execucao))
)

data_limite = (
    filmes_lancados_validos
    .agg(F.max("data_lancamento").alias("data_limite"))
    .first()["data_limite"]
)

if data_limite is None:
    raise ValueError(
        "Não existem lançamentos válidos para definir o período."
    )

# Início exclusivo e fim inclusivo.
filmes_ultimos_2_anos = (
    filmes_lancados_validos
    .filter(
        (
            F.col("data_lancamento")
            > F.add_months(F.lit(data_limite), -24)
        )
        & (F.col("data_lancamento") <= F.lit(data_limite))
    )
    .select("sk_movie_id")
)

participacoes_atores = (
    ponte_pessoas_gold
    .join(
        filmes_ultimos_2_anos,
        on="sk_movie_id",
        how="inner",
    )
    .join(
        pessoas_gold.filter(F.col("tipo_pessoa") == "Ator"),
        on="sk_person_id",
        how="inner",
    )
    .groupBy("sk_person_id", "nome_pessoa")
    .agg(
        F.countDistinct("sk_movie_id")
        .alias("quantidade_participacoes")
    )
)

# Preservamos todos os atores empatados na liderança.
df_atores_lideres = (
    participacoes_atores
    .withColumn(
        "posicao",
        F.rank().over(
            Window.orderBy(F.desc("quantidade_participacoes"))
        ),
    )
    .filter(F.col("posicao") == 1)
    .select(
        F.col("nome_pessoa").alias("ator"),
        "quantidade_participacoes",
        F.add_months(
            F.lit(data_limite), -24
        ).alias("inicio_exclusivo"),
        F.lit(data_limite).alias("fim_inclusivo"),
    )
    .orderBy("ator")
)

display(df_atores_lideres)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


ator,quantidade_participacoes,inicio_exclusivo,fim_inclusivo
Kevin Hart,64,2024-02-19,2026-02-19


#### 6. Qual produtora teve o maior lucro nos filmes lançados nos últimos cinco anos?

Universal Pictures liderou com R$ 29.767.326.921,64 de lucro acumulado conhecido nos filmes associados, considerando lançamentos posteriores a 19/02/2021 e até 19/02/2026, inclusive. A produtora possui 50 filmes vinculados nesse período, dos quais 24 apresentam lucro informado e 26 não possuem informação suficiente para calculá-lo. Portanto, o resultado representa apenas os valores disponíveis. O lucro integral de cada filme foi atribuído às produtoras associadas, sem divisão por participação financeira; assim, esse valor não representa o lucro contábil individual da empresa.

In [0]:
# 6. Qual produtora teve o maior lucro nos últimos cinco anos?
# Utilizamos os lançamentos válidos e a data_limite
# definidos na pergunta 5.

produtoras_gold = spark.table(
    f"{catalogo}.{schema_gold}.dim_companies"
)

ponte_produtoras_gold = spark.table(
    f"{catalogo}.{schema_gold}.bridge_movie_company"
)

filmes_ultimos_5_anos = (
    filmes_lancados_validos
    .filter(
        (
            F.col("data_lancamento")
            > F.add_months(F.lit(data_limite), -60)
        )
        & (F.col("data_lancamento") <= F.lit(data_limite))
    )
    .select("sk_movie_id")
)

# Sem informação sobre participação financeira, atribuímos
# o lucro integral do filme a cada produtora associada.
lucro_por_produtora = (
    ponte_produtoras_gold
    .join(
        filmes_ultimos_5_anos,
        on="sk_movie_id",
        how="inner",
    )
    .join(
        fato_gold.select("sk_movie_id", "lucro_brl"),
        on="sk_movie_id",
        how="inner",
    )
    .join(
        produtoras_gold,
        on="sk_company_id",
        how="inner",
    )
    .groupBy("sk_company_id", "nome_produtora")
    .agg(
        F.sum("lucro_brl").alias("lucro_total_brl"),
        F.countDistinct("sk_movie_id")
        .alias("filmes_associados_no_periodo"),
        F.count("lucro_brl").alias("filmes_com_lucro_informado"),
        F.count(
            F.when(F.col("lucro_brl").isNull(), 1)
        ).alias("filmes_sem_lucro_informado"),
    )
)

# Valores negativos participam da soma.
# Sem nenhum lucro conhecido, a produtora não entra no ranking.
df_produtoras_lideres = (
    lucro_por_produtora
    .filter(F.col("lucro_total_brl").isNotNull())
    .withColumn(
        "posicao",
        F.rank().over(
            Window.orderBy(F.desc("lucro_total_brl"))
        ),
    )
    .filter(F.col("posicao") == 1)
    .select(
        "nome_produtora",
        "lucro_total_brl",
        "filmes_associados_no_periodo",
        "filmes_com_lucro_informado",
        "filmes_sem_lucro_informado",
        F.add_months(
            F.lit(data_limite), -60
        ).alias("inicio_exclusivo"),
        F.lit(data_limite).alias("fim_inclusivo"),
    )
    .orderBy("nome_produtora")
)

display(df_produtoras_lideres)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


nome_produtora,lucro_total_brl,filmes_associados_no_periodo,filmes_com_lucro_informado,filmes_sem_lucro_informado,inicio_exclusivo,fim_inclusivo
Universal Pictures,29767326921.64,50,24,26,2021-02-19,2026-02-19


## Reexecução e conclusão da camada Gold

Para reconstruir esta camada, executaremos o notebook desde o início, utilizando as tabelas Silver já salvas. As dimensões e seus relacionamentos devem ser gerados juntos, pois a inclusão de novos cadastros pode alterar as chaves numéricas. A gravação substitui as dez tabelas Gold, evitando acumular registros entre execuções. Como as tabelas são gravadas separadamente, uma execução interrompida deverá ser concluída antes de consumir o conjunto atualizado. 

A camada organiza os dados para análise e prepara os documentos de contexto para o futuro assistente de IA, sem implementar o assistente ou a vetorização. Os resultados permanecem sujeitos à cobertura financeira, aos vínculos disponíveis e às limitações de qualidade registradas na Silver. A próxima etapa do projeto será configurar e testar o Job com a sequência Bronze, Silver e Gold.